In [1]:
import os
import re
from pathlib import Path

import kagglehub
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("KAGGLE_API_TOKEN"):
    raise RuntimeError("KAGGLE_API_TOKEN was not loaded")

In [2]:
train_path = kagglehub.competition_download(
    "machine-learning-task-tdt-4173-2026", path="sample_submission.csv"
)
train_df = pd.read_csv(train_path)
train_df.head()

,Run No,result_committed_Hogga_G1_t0,result_committed_Hogga_G1_t1,result_committed_Hogga_G1_t2,result_committed_Hogga_G1_t3,result_committed_Hogga_G1_t4,result_committed_Hogga_G1_t5,result_committed_Hogga_G1_t6,result_committed_Hogga_G1_t7,result_committed_Hogga_G1_t8,...,result_committed_Vesle Kjela_G1_t158,result_committed_Vesle Kjela_G1_t159,result_committed_Vesle Kjela_G1_t160,result_committed_Vesle Kjela_G1_t161,result_committed_Vesle Kjela_G1_t162,result_committed_Vesle Kjela_G1_t163,result_committed_Vesle Kjela_G1_t164,result_committed_Vesle Kjela_G1_t165,result_committed_Vesle Kjela_G1_t166,result_committed_Vesle Kjela_G1_t167
0,2923,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2924,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2925,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2926,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2927,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
for path in Path("data").rglob("*"):
    if ".csv" in path.suffix.lower():
        print(path)

data/extended/Constraint_min_volume.csv
data/extended/Constraint_min_flow.csv
data/kernel/Historical_day_ahead_price_2015_2025.csv
data/kernel/Historical_volume_2015_2024.csv
data/kernel/Historical_inflow_1958_2025.csv
data/kernel/Unit_commitment_decisions.csv
data/kernel/Synthetic_water_value_2015_2024.csv


In [4]:
DATA_DIR = Path("data")

paths = {
    "price": DATA_DIR / "kernel/Historical_day_ahead_price_2015_2025.csv",
    "volume": DATA_DIR / "kernel/Historical_volume_2015_2024.csv",
    "inflow": DATA_DIR / "kernel/Historical_inflow_1958_2025.csv",
    "target": DATA_DIR / "kernel/Unit_commitment_decisions.csv",
    "water_value": DATA_DIR / "kernel/Synthetic_water_value_2015_2024.csv",
    "min_volume": DATA_DIR / "extended/Constraint_min_volume.csv",
    "min_flow": DATA_DIR / "extended/Constraint_min_flow.csv",
}

raw = {}

for name, path in paths.items():
    raw[name] = pd.read_csv(path)

    print(f"{name:12s}", raw[name].shape, list(raw[name].columns))

price        (103062, 2) ['date', 'Day-ahead price (EUR/MWh)']
volume       (3654, 18) ['date', 'Kjelavatn', 'Staavatn', 'Langesae', 'Foersvatn', 'Venemo', 'Botnedalsvatn', 'Bitdalsvatn', 'Vaamarvatn', 'Langeidvatn', 'Totak', 'Songavatn', 'Vinjevatn', 'Bandak', 'Bordalsvatn', 'Byrtevatn', 'Hyljelihyl', 'Vatjern']
inflow       (594224, 19) ['date', 'Langesae', 'Staavatn', 'Kjelavatn', 'Bordalsvatn', 'Foersvatn', 'Hyljelihyl', 'Venemo', 'Bitdalsvatn', 'Songavatn', 'Totak', 'Vaamarvatn', 'Langeidvatn', 'Vatjern', 'Vinjevatn', 'Botnedalsvatn', 'Byrtevatn', 'Bandak', 'r_Vest_Vassdraget']
target       (2922, 2355) ['Run No', 'starttime', 'total_calculation_time', 'result_committed_Hogga_G1_t0', 'result_committed_Hogga_G1_t1', 'result_committed_Hogga_G1_t2', 'result_committed_Hogga_G1_t3', 'result_committed_Hogga_G1_t4', 'result_committed_Hogga_G1_t5', 'result_committed_Hogga_G1_t6', 'result_committed_Hogga_G1_t7', 'result_committed_Hogga_G1_t8', 'result_committed_Hogga_G1_t9', 'result_commit

In [5]:
target = pd.read_csv(paths["target"])

print(target.head())
print(target.columns.tolist())

   Run No   starttime  total_calculation_time  result_committed_Hogga_G1_t0  \
0       1  2015-01-01               48.950692                             1   
1       2  2015-01-02               56.048899                             1   
2       3  2015-01-03               44.280307                             1   
3       4  2015-01-04               40.848453                             1   
4       5  2015-01-05               44.036279                             1   

   result_committed_Hogga_G1_t1  result_committed_Hogga_G1_t2  \
0                             1                             1   
1                             1                             1   
2                             1                             1   
3                             1                             1   
4                             1                             1   

   result_committed_Hogga_G1_t3  result_committed_Hogga_G1_t4  \
0                             1                             1   
1   

In [ ]:
def normalize_columns(df):
    df = df.copy()
    df.columns = [
        re.sub(r"[^a-zA-Z0-9]+", "_", str(column).strip().lower()).strip("_")
        for column in df.columns
    ]
    return df


data = {name: normalize_columns(df) for name, df in raw.items()}

feature_tables = []
for name, df in data.items():
    if name == "target":
        continue

    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])

    # Keep date unchanged, prefix all other columns
    df = df.rename(
        columns={
            column: f"{name}_{column}" for column in df.columns if column != "date"
        }
    )

    feature_tables.append(df)
features = feature_tables[0]

for df in feature_tables[1:]:
    features = features.merge(df, on="date", how="outer")

features = features.sort_values("date")
features

,date,price_day_ahead_price_eur_mwh,volume_kjelavatn,volume_staavatn,volume_langesae,volume_foersvatn,volume_venemo,volume_botnedalsvatn,volume_bitdalsvatn,volume_vaamarvatn,...,water_value_vatjern,min_volume_totak,min_volume_byrtevatn,min_volume_staavatn,min_flow_b_kjelavatn,min_flow_b_hyljelihyl,min_flow_b_vinjevatn,min_flow_b_byrtevatn,min_flow_b_bandak,min_flow_r_vest_vassdraget
0,1958-01-01 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1958-01-01 01:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1958-01-01 02:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1958-01-01 03:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1958-01-01 04:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
602713,2025-12-31 22:45:00+00:00,65.31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
602714,2025-12-31 23:00:00+00:00,65.60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0.3,0.5,1.0,0.0,4.0,5.0
602715,2025-12-31 23:15:00+00:00,64.61,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
602716,2025-12-31 23:30:00+00:00,63.31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
target = data["target"].copy()

print(target.shape)
print(target.columns.tolist())
print(target.head())

(2922, 2355)
['run_no', 'starttime', 'total_calculation_time', 'result_committed_hogga_g1_t0', 'result_committed_hogga_g1_t1', 'result_committed_hogga_g1_t2', 'result_committed_hogga_g1_t3', 'result_committed_hogga_g1_t4', 'result_committed_hogga_g1_t5', 'result_committed_hogga_g1_t6', 'result_committed_hogga_g1_t7', 'result_committed_hogga_g1_t8', 'result_committed_hogga_g1_t9', 'result_committed_hogga_g1_t10', 'result_committed_hogga_g1_t11', 'result_committed_hogga_g1_t12', 'result_committed_hogga_g1_t13', 'result_committed_hogga_g1_t14', 'result_committed_hogga_g1_t15', 'result_committed_hogga_g1_t16', 'result_committed_hogga_g1_t17', 'result_committed_hogga_g1_t18', 'result_committed_hogga_g1_t19', 'result_committed_hogga_g1_t20', 'result_committed_hogga_g1_t21', 'result_committed_hogga_g1_t22', 'result_committed_hogga_g1_t23', 'result_committed_hogga_g1_t24', 'result_committed_hogga_g1_t25', 'result_committed_hogga_g1_t26', 'result_committed_hogga_g1_t27', 'result_committed_hogga

In [8]:
training_data = target.merge(features, on="date", how="left")

KeyError: 'date'